# Muraqam (مُرقّم) — two-stage cascade: **dots first, then the rest**

Single plain AraBERT (`aubmindlab/bert-base-arabertv2`), the idea from the chat done as a **two-model cascade**:

- **Stage A — dots only.** A model whose whole job is `.` (where sentences end). One output, high signal.
- **Insert** the predicted dots into the text (`word` → `word.`), word count preserved.
- **Stage B — the rest.** A second model trained to predict `،؟!:؛-` **on dotted input**. Because the dot lives in B's *input*, not at its *output*, B never has to choose between `.` and a rare mark — which is why this avoids the suppression that a single shared head suffers.

**Why B is trained on dots (not just fed them at inference):** a head fine-tuned on bare text treats injected dots as OOD. Here B is trained with **noisy gold dots** (`DOTNOISE_KEEP`/`DOTNOISE_ADD` simulate Stage A's real recall/false-positives), so it learns to *use* the boundaries and stays robust to A's mistakes.

**Final marks** = Stage A dots (`.` column) + Stage B six columns → assembled into the 7-mark vector, honorific `-` rule applied, thresholds tuned on the assembled probs, then scored with the host metric.

**Read the run:** `TRAIN_SINGLE_BASELINE=True` trains the original 7-way single model too, so you get a same-run CV delta: baseline vs cascade, with per-class F1 side by side. Watch `؛ ! ؟ :`.

> Requires **internet ON** and a **GPU**.


In [ ]:
# =========================================================================
#  Muraqam (مُرقّم) — Arabic Punctuation Restoration
#  TWO-STAGE CASCADE:  Stage A predicts dots -> insert -> Stage B predicts rest.
#  Metric: macro-F1 over 7 marks ( . ، ؟ ! : ؛ - ), multi-label per gap.
# =========================================================================
!pip install -q transformers torch

import os, re, random, math, json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel

SEED = 2026
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"

# --- data paths (Kaggle) ---
TRAIN_CSV = "/kaggle/input/competitions/muraqqamchallenge/train.csv"   # adjust to your slug
TEST_CSV  = "/kaggle/input/competitions/muraqqamchallenge/test.csv"
if not os.path.exists(TRAIN_CSV):
    TRAIN_CSV = "train.csv"; TEST_CSV = "test.csv"

# --- the 7 scored marks, fixed canonical order ---
MARKS = ['.', '،', '؟', '!', ':', '؛', '-']
M2I = {m: i for i, m in enumerate(MARKS)}
NUM_MARKS = len(MARKS)

# --- single PLAIN AraBERT backbone for BOTH stages (as requested) ---
BACKBONE = "aubmindlab/bert-base-arabertv2"

# --- two-stage mark split ---
MARKS_A = ['.']                                   # Stage A: dots only
MARKS_B = ['،', '؟', '!', ':', '؛', '-']          # Stage B: the rest, given dots
DOT_IDX  = [M2I[m] for m in MARKS_A]              # -> [0]
REST_IDX = [M2I[m] for m in MARKS_B]              # -> [1,2,3,4,5,6]

# --- Stage B training: inject NOISY gold dots so B is robust to Stage A errors ---
DOTNOISE_KEEP = 0.92     # keep a gold dot (simulate Stage-A recall)
DOTNOISE_ADD  = 0.02     # add a spurious dot (simulate Stage-A false positives)

# --- training config ---
CHUNK_WORDS = 180
STRIDE      = 140
MAX_LEN     = 320
BATCH       = 8
EPOCHS      = 20
LR          = 2e-5
FOCAL_GAMMA = 2.0
VAL_FRAC    = 0.10
USE_AMP             = True
EARLY_STOP_PATIENCE = 5
MAX_EPOCHS_CAP      = 20

# --- honorifics wrapped as -X- (rule-handled) ---
HONORIFICS = {"ﷺ"}
FORCE_HONORIFIC_RULE = True

# --- also train the original single 7-way model for a same-run CV delta ---
TRAIN_SINGLE_BASELINE = True

print("device:", device, "| backbone:", BACKBONE)
print("Stage A marks:", MARKS_A, "| Stage B marks:", MARKS_B, "| baseline:", TRAIN_SINGLE_BASELINE)


## 1. Host metric (verbatim)

In [ ]:
# Host metric (verbatim) — for trustworthy LOCAL validation before you submit.
class ParticipantVisibleError(Exception):
    pass

"""
Kaggle metric for Arabic Punctuation Restoration.

Conventions:
    - `solution` is the full test CSV, including the hidden gold column.
    - `submission` is the competitor's CSV.
    - Both have a row_id column (already aligned & sorted by Kaggle).
    - `solution` has a column `raw` with the unpunctuated input and a column
      `gold` with the reference punctuated string.
    - `submission` has a column `prediction` with the competitor's
      punctuated string.

Scoring:
    Macro-F1 over the 7 Arabic sentence-punctuation classes
    ( . ، ؟ ! : ؛ - ), EXCLUDING the "no punctuation" class.

"""

import re
from typing import Optional

import pandas as pd
from sklearn.metrics import f1_score
from sklearn.preprocessing import MultiLabelBinarizer

# ---------------------------------------------------------------------------
# Classical Arabic punctuation-restoration set. Anything outside this is
# treated as word content (e.g. parentheses, quotes, numbers) and is NOT
# scored. Competitors must preserve those characters in their predictions.
# ---------------------------------------------------------------------------
VALID_SYMBOLS = set('.،؟!:؛-')


def _tokenize_gold(text: str):
    """
    Split a (gold or prediction) string into (leading_gap, [(word, trailing_gap), ...]).
    A "word" is a maximal run of non-whitespace, non-whitelist chars.
    A "gap" is any run of whitelist chars between words.
    """
    leading_gap_chars = []
    pairs = []
    current_word_chars = []
    in_word = False

    for ch in text:
        if ch.isspace():
            if in_word:
                pairs.append([''.join(current_word_chars), []])
                current_word_chars = []
                in_word = False
            continue

        if ch in VALID_SYMBOLS:
            if in_word:
                pairs.append([''.join(current_word_chars), [ch]])
                current_word_chars = []
                in_word = False
            else:
                if pairs:
                    pairs[-1][1].append(ch)
                else:
                    leading_gap_chars.append(ch)
            continue

        if not in_word:
            in_word = True
            current_word_chars = [ch]
        else:
            current_word_chars.append(ch)

    if in_word:
        pairs.append([''.join(current_word_chars), []])

    return ''.join(leading_gap_chars), [(w, ''.join(g)) for (w, g) in pairs]


def _extract_labels(raw_text: str, generated_text: str, role: str):
    """
    Align `generated_text` to `raw_text` word-by-word and return one list of
    symbols per word, representing the punctuation that appears in the gap
    after each word.

    Raises ValueError on any structural mismatch.
    """
    if raw_text is None or generated_text is None:
        raise ValueError(f"[{role}] text is empty or null")

    raw_words = str(raw_text).strip().split()
    if not raw_words:
        raise ValueError(f"[{role}] raw text contains no words")

    _, pairs = _tokenize_gold(str(generated_text))
    gen_words = [w for (w, _) in pairs]

    if len(gen_words) != len(raw_words):
        raise ValueError(
            f"[{role}] word-count mismatch: raw has {len(raw_words)} words, "
            f"{role} has {len(gen_words)}"
        )

    for i, (rw, gw) in enumerate(zip(raw_words, gen_words)):
        if rw != gw:
            raise ValueError(
                f"[{role}] word mismatch at position {i}: "
                f"raw='{rw}' vs {role}='{gw}'"
            )

    labels = []
    for _, gap in pairs:
        syms = [c for c in gap if c in VALID_SYMBOLS]
        labels.append(syms if syms else ['0'])

    return labels


def score(
    solution: pd.DataFrame,
    submission: pd.DataFrame,
    row_id_column_name: str,
    raw_column_name: str = 'text',
    gold_column_name: str = 'final_text',
    prediction_column_name: str = 'final_text',
) -> float:
    """
    Returns macro-F1 over the 7 Arabic punctuation
    classes, excluding the "no punctuation" class.
    """
    # --- 0. Drop row_id; Kaggle has already aligned the two frames -----------
    del solution[row_id_column_name]
    del submission[row_id_column_name]

    # --- 1. Column presence -------------------------------------------------
    for col in (raw_column_name, gold_column_name):
        if col not in solution.columns:
            # Organizer-side problem — hidden from competitor.
            raise RuntimeError(f"Solution is missing column '{col}'")
    if prediction_column_name not in submission.columns:
        raise ParticipantVisibleError(
            f"Submission is missing column '{prediction_column_name}'"
        )

    # --- 2. Length match ----------------------------------------------------
    if len(solution) != len(submission):
        raise ParticipantVisibleError(
            f"Submission has {len(submission)} rows, expected {len(solution)}"
        )

    # --- 3. Extract labels row-by-row --------------------------------------
    true_labels = []
    pred_labels = []

    raws = solution[raw_column_name].tolist()
    golds = solution[gold_column_name].tolist()
    preds = submission[prediction_column_name].tolist()

    for idx, (raw, gold, pred) in enumerate(zip(raws, golds, preds)):
        try:
            gold_seq = _extract_labels(raw, gold, role="gold")
        except ValueError as e:
            # Organizer-side: our own gold CSV is malformed for this row.
            raise RuntimeError(
                f"Gold extraction failed on row index {idx}: {e}"
            ) from e

        try:
            pred_seq = _extract_labels(raw, pred, role="prediction")
        except ValueError as e:
            # Competitor can fix this themselves.
            raise ParticipantVisibleError(
                f"Prediction at row index {idx} does not align with the raw "
                f"input. Your prediction must contain the same sequence of "
                f"non-punctuation words as the input, with only the allowed "
                f"punctuation symbols {sorted(VALID_SYMBOLS)} inserted "
                f"between them. Details: {e}"
            ) from e

        if len(gold_seq) != len(pred_seq):
            raise ParticipantVisibleError(
                f"Row {idx}: prediction has {len(pred_seq)} word positions "
                f"but the input has {len(gold_seq)}"
            )

        true_labels.extend(gold_seq)
        pred_labels.extend(pred_seq)

    # --- 4. Binarize using gold ∪ pred so hallucinated marks cost precision --
    all_classes = sorted(VALID_SYMBOLS) + ['0']
    mlb = MultiLabelBinarizer(classes=all_classes)
    y_true = mlb.fit_transform(true_labels)
    y_pred = mlb.transform(pred_labels)

    classes = list(mlb.classes_)
    zero_idx = classes.index('0')
    scored_cols = [i for i in range(len(classes)) if i != zero_idx]

    return float(f1_score(
        y_true[:, scored_cols],
        y_pred[:, scored_cols],
        average='macro',
        zero_division=0,
    ))

## 2. Word-gap tokenizer, labeling & honorific rule

In [ ]:
# =========================================================================
#  Word-gap tokenizer (IDENTICAL to the host metric) + labeling + rules
#  Sharing the metric's tokenizer guarantees zero train/scoring gap.
# =========================================================================
VALID_SYMBOLS = set('.،؟!:؛-')

def tokenize_gold(text):
    """-> (leading_gap, [(word, trailing_gap), ...]).  Matches host metric."""
    leading=[]; pairs=[]; cur=[]; in_word=False
    for ch in str(text):
        if ch.isspace():
            if in_word: pairs.append([''.join(cur), []]); cur=[]; in_word=False
            continue
        if ch in VALID_SYMBOLS:
            if in_word: pairs.append([''.join(cur), [ch]]); cur=[]; in_word=False
            else:
                if pairs: pairs[-1][1].append(ch)
                else: leading.append(ch)
            continue
        if not in_word: in_word=True; cur=[ch]
        else: cur.append(ch)
    if in_word: pairs.append([''.join(cur), []])
    return ''.join(leading), [(w, ''.join(g)) for w,g in pairs]

def gold_to_labels(final_text):
    """Return (words, Y) where Y is (n_words, NUM_MARKS) multi-hot of the gap AFTER each word."""
    _, pairs = tokenize_gold(final_text)
    words=[w for w,_ in pairs]
    Y=np.zeros((len(words), NUM_MARKS), dtype=np.float32)
    for i,(_,gap) in enumerate(pairs):
        for c in gap:
            if c in M2I: Y[i, M2I[c]] = 1.0
    return words, Y

def words_of_raw(raw):
    """Word units exactly as the metric derives them: whitespace split."""
    return str(raw).strip().split()

MARK_ORDER = MARKS  # canonical write order inside a gap (scoring is set-based anyway)

def reconstruct(words, pred_multi):
    """words + per-word multi-hot -> final_text string (word count preserved)."""
    toks=[]
    for w, row in zip(words, pred_multi):
        marks=''.join(m for m in MARK_ORDER if row[M2I[m]]>0)
        toks.append(w+marks)
    return ' '.join(toks)

def apply_honorific_rule(words, pred_multi):
    """Force -X- around each honorific: dash on its own gap AND the previous word's gap."""
    if not FORCE_HONORIFIC_RULE: return pred_multi
    P=pred_multi.copy()
    di=M2I['-']
    for i,w in enumerate(words):
        if w in HONORIFICS:
            P[i, di]=1.0
            if i>0: P[i-1, di]=1.0
    return P

# ---- load + split ----
df = pd.read_csv(TRAIN_CSV)
df = df.dropna(subset=["text","final_text"]).reset_index(drop=True)
idx = np.arange(len(df)); rng=np.random.default_rng(SEED); rng.shuffle(idx)
n_val=max(1,int(len(df)*VAL_FRAC))
val_idx=set(idx[:n_val].tolist())
train_rows=[df.iloc[i] for i in range(len(df)) if i not in val_idx]
val_rows  =[df.iloc[i] for i in range(len(df)) if i in val_idx]
print(f"train rows: {len(train_rows)} | val rows: {len(val_rows)}")

# sanity: gold words must equal raw words (they do, per analysis)
_bad=0
for r in train_rows+val_rows:
    w_gold,_=gold_to_labels(r["final_text"]); 
    if w_gold!=words_of_raw(r["text"]): _bad+=1
print(f"word-alignment mismatches (want 0): {_bad}")

## 3. Sliding-window dataset

In [ ]:
# =========================================================================
#  Sliding-window dataset, generalized for the cascade.
#   - target_idx : which columns of the 7-mark Y this model predicts
#                  (Stage A -> [0]; Stage B -> [1..6]; baseline -> all 7)
#   - inject_dots: Stage B only. Append NOISY gold dots to input words so B
#                  trains on dotted text (word -> word.). Word COUNT preserved,
#                  so host-metric alignment and label indexing stay exact.
# =========================================================================
def chunk_words(words, Y=None):
    n=len(words)
    if n<=CHUNK_WORDS:
        yield words, (Y if Y is not None else None), 0; return
    s=0
    while s<n:
        e=min(s+CHUNK_WORDS, n)
        yield words[s:e], (Y[s:e] if Y is not None else None), s
        if e==n: break
        s+=STRIDE

def inject_gold_dots(words, Yfull, keep, add):
    """Append gold '.' to a word with prob `keep`; add spurious '.' with prob `add`."""
    di=M2I['.']; out=[]
    for i,w in enumerate(words):
        has = Yfull[i,di] > 0
        if has and random.random()<keep:            out.append(w+'.')
        elif (not has) and random.random()<add:      out.append(w+'.')
        else:                                        out.append(w)
    return out

class PunctDataset(Dataset):
    def __init__(self, rows, tokenizer, target_idx, inject_dots=False, has_labels=True):
        self.samples=[]; self.tok=tokenizer; self.has_labels=has_labels
        self.target_idx=list(target_idx); self.n_out=len(self.target_idx)
        self.inject_dots=inject_dots
        self.max_len=MAX_LEN + (48 if inject_dots else 0)   # headroom for dot subwords
        for r in rows:
            words=words_of_raw(r["text"])
            Yfull = gold_to_labels(r["final_text"])[1] if has_labels else None
            for wslice,Yslice,start in chunk_words(words, Yfull):
                self.samples.append((wslice, Yslice))
    def __len__(self): return len(self.samples)
    def __getitem__(self,i):
        words,Yfull=self.samples[i]
        enc_words=words
        if self.inject_dots and Yfull is not None:
            enc_words=inject_gold_dots(words, Yfull, DOTNOISE_KEEP, DOTNOISE_ADD)
        enc=self.tok(enc_words, is_split_into_words=True, truncation=True,
                     max_length=self.max_len, return_tensors=None)
        word_ids=enc.word_ids()
        last_pos={}
        for pos,wid in enumerate(word_ids):
            if wid is not None: last_pos[wid]=pos
        active=np.zeros(len(word_ids),dtype=bool)
        labels=np.zeros((len(word_ids),self.n_out),dtype=np.float32)
        wid_at=np.full(len(word_ids),-1,dtype=np.int64)
        for wid,pos in last_pos.items():
            active[pos]=True; wid_at[pos]=wid
            if Yfull is not None and wid<len(Yfull):
                labels[pos]=Yfull[wid, self.target_idx]
        return {"input_ids":enc["input_ids"],"attention_mask":enc["attention_mask"],
                "active":active,"labels":labels,"word_ids":wid_at}

def collate(batch, pad_id):
    maxlen=max(len(b["input_ids"]) for b in batch); B=len(batch)
    n_out=batch[0]["labels"].shape[1]
    input_ids=np.full((B,maxlen),pad_id,dtype=np.int64)
    attn=np.zeros((B,maxlen),dtype=np.int64)
    active=np.zeros((B,maxlen),dtype=bool)
    labels=np.zeros((B,maxlen,n_out),dtype=np.float32)
    wids=np.full((B,maxlen),-1,dtype=np.int64)
    for i,b in enumerate(batch):
        L=len(b["input_ids"])
        input_ids[i,:L]=b["input_ids"]; attn[i,:L]=b["attention_mask"]
        active[i,:L]=b["active"]; labels[i,:L]=b["labels"]; wids[i,:L]=b["word_ids"]
    return (torch.tensor(input_ids),torch.tensor(attn),torch.tensor(active),
            torch.tensor(labels),torch.tensor(wids))
print("dataset utilities ready (cascade)")


## 4. Multi-label model + focal loss
(`PunctModel` uses `AutoModel.from_pretrained`, which loads the warm-start encoder and discards its token-classification head.)

In [ ]:
# =========================================================================
#  Multi-LABEL token classifier (independent sigmoid per mark) + focal loss.
#  Now parameterized by n_out so one class can drive Stage A (1 out), Stage B
#  (6 out) and the 7-way baseline from the same code.
# =========================================================================
class PunctModel(nn.Module):
    def __init__(self, model_name, n_out):
        super().__init__()
        self.backbone=AutoModel.from_pretrained(model_name)
        h=self.backbone.config.hidden_size
        self.drop=nn.Dropout(0.1)
        self.head=nn.Linear(h,n_out)
    def forward(self,input_ids,attention_mask):
        out=self.backbone(input_ids=input_ids,attention_mask=attention_mask).last_hidden_state
        return self.head(self.drop(out))          # (B,T,n_out) logits

def focal_bce(logits, targets, active, gamma=FOCAL_GAMMA, pos_weight=None):
    """Focal binary cross-entropy over ACTIVE positions only."""
    logits=logits[active]; targets=targets[active]
    if logits.numel()==0:
        return logits.sum()*0.0
    bce=nn.functional.binary_cross_entropy_with_logits(
        logits,targets,reduction='none',pos_weight=pos_weight)
    p=torch.sigmoid(logits)
    p_t=p*targets+(1-p)*(1-targets)
    focal=((1-p_t)**gamma)*bce
    return focal.mean()

def compute_pos_weight(rows, target_idx):
    """Per-mark positive weight from class frequency, restricted to target_idx."""
    ti=list(target_idx); pos=np.zeros(len(ti)); tot=0
    for r in rows:
        _,Y=gold_to_labels(r["final_text"]); pos+=Y[:,ti].sum(0); tot+=len(Y)
    neg=tot-pos
    w=np.clip(neg/np.clip(pos,1,None),1.0,20.0)
    return torch.tensor(w,dtype=torch.float32)
print("model + focal loss ready (n_out parameterized)")


## 5. Training (AMP + early stopping) & windowed inference

In [ ]:
# =========================================================================
#  Training (AMP + early stopping) generalized over target_idx / inject_dots,
#  plus windowed inference and the TWO-STAGE cascade predictor.
# =========================================================================
from functools import partial

def _val_macro_f1(model, tok, val_rows, target_idx, inject_dots=False):
    """Per-word macro-F1 at 0.5 over the model's OWN target classes (early-stop
    signal). For Stage B, inject CLEAN gold dots so val matches its input dist."""
    model.eval(); ti=list(target_idx); no=len(ti)
    tp=np.zeros(no); fp=np.zeros(no); fn=np.zeros(no)
    ml=MAX_LEN+(48 if inject_dots else 0)
    with torch.no_grad():
        for r in val_rows:
            words=words_of_raw(r["text"]); n=len(words)
            Yf=gold_to_labels(r["final_text"])[1]; Y=Yf[:,ti]
            enc_words=inject_gold_dots(words,Yf,1.0,0.0) if inject_dots else words
            acc=np.zeros((n,no)); cnt=np.zeros((n,1))+1e-9
            for s,e in _chunk_bounds(n):
                enc=tok(enc_words[s:e],is_split_into_words=True,truncation=True,max_length=ml,return_tensors="pt")
                wid=enc.word_ids()
                logits=model(enc["input_ids"].to(device),enc["attention_mask"].to(device))[0]
                probs=torch.sigmoid(logits).float().cpu().numpy()
                last={}
                for pos,w in enumerate(wid):
                    if w is not None: last[w]=pos
                for w,pos in last.items():
                    gi=s+w
                    if gi<n: acc[gi]+=probs[pos]; cnt[gi]+=1
            pred=((acc/cnt)>=0.5).astype(int)
            tp+=((pred==1)&(Y==1)).sum(0); fp+=((pred==1)&(Y==0)).sum(0); fn+=((pred==0)&(Y==1)).sum(0)
    prec=tp/np.clip(tp+fp,1,None); rec=tp/np.clip(tp+fn,1,None)
    f1=np.where((prec+rec)>0,2*prec*rec/np.clip(prec+rec,1e-9,None),0.0)
    return float(f1.mean())

def train_one(model_name, train_rows, val_rows, target_idx, inject_dots=False, tag=""):
    tok=AutoTokenizer.from_pretrained(model_name)
    pad_id=tok.pad_token_id if tok.pad_token_id is not None else 0
    tr=PunctDataset(train_rows,tok,target_idx,inject_dots=inject_dots,has_labels=True)
    dl=DataLoader(tr,batch_size=BATCH,shuffle=True,collate_fn=partial(collate,pad_id=pad_id))
    model=PunctModel(model_name,len(target_idx)).to(device)
    pw=compute_pos_weight(train_rows,target_idx).to(device)
    opt=torch.optim.AdamW(model.parameters(),lr=LR)
    n_epochs=MAX_EPOCHS_CAP; total=len(dl)*n_epochs
    sched=torch.optim.lr_scheduler.OneCycleLR(opt,max_lr=LR,total_steps=total,pct_start=0.1)
    scaler=torch.amp.GradScaler('cuda',enabled=(USE_AMP and device=="cuda"))

    best_f1=-1.0; best_state=None; patience=0
    for ep in range(n_epochs):
        model.train(); run=0.0
        for input_ids,attn,active,labels,_ in dl:
            input_ids,attn=input_ids.to(device),attn.to(device)
            active,labels=active.to(device),labels.to(device)
            opt.zero_grad()
            with torch.amp.autocast('cuda',enabled=(USE_AMP and device=="cuda")):
                logits=model(input_ids,attn)
                loss=focal_bce(logits,labels,active,pos_weight=pw)
            scaler.scale(loss).backward()
            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
            scaler.step(opt); scaler.update(); sched.step(); run+=loss.item()
        vf1=_val_macro_f1(model,tok,val_rows,target_idx,inject_dots=inject_dots) if val_rows else -1.0
        line=f"  [{tag}] epoch {ep+1}/{n_epochs} loss {run/len(dl):.4f}"
        if val_rows:
            line+=f" | val F1 {vf1:.4f}"
            if vf1>best_f1+1e-4:
                best_f1=vf1; best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}; patience=0; line+="  *"
            else: patience+=1
        print(line)
        if val_rows and patience>=EARLY_STOP_PATIENCE:
            print(f"    early stop; best F1 {best_f1:.4f}"); break
    if best_state is not None: model.load_state_dict(best_state)
    return model, tok

# ---------------------------------------------------------------------------
#  Windowed inference (n_out-aware). enc_words_list overrides the encoder input.
# ---------------------------------------------------------------------------
def _chunk_bounds(n):
    if n<=CHUNK_WORDS:
        yield 0,n; return
    s=0
    while s<n:
        e=min(s+CHUNK_WORDS,n); yield s,e
        if e==n: break
        s+=STRIDE

@torch.no_grad()
def _infer_rows(model, tok, rows, n_out, enc_words_list=None):
    model.eval(); results=[]
    for ri,r in enumerate(rows):
        words=words_of_raw(r["text"]); n=len(words)
        enc_words=enc_words_list[ri] if enc_words_list is not None else words
        ml=MAX_LEN+(48 if enc_words_list is not None else 0)
        acc=np.zeros((n,n_out)); cnt=np.zeros((n,1))+1e-9
        for s,e in _chunk_bounds(n):
            enc=tok(enc_words[s:e],is_split_into_words=True,truncation=True,max_length=ml,return_tensors="pt")
            wid=enc.word_ids()
            logits=model(enc["input_ids"].to(device),enc["attention_mask"].to(device))[0]
            probs=torch.sigmoid(logits).float().cpu().numpy()
            last={}
            for pos,w in enumerate(wid):
                if w is not None: last[w]=pos
            for w,pos in last.items():
                gi=s+w
                if gi<n: acc[gi]+=probs[pos]; cnt[gi]+=1
        results.append((words, acc/cnt))
    return results

def predict_probs(model, tok, rows):
    """Baseline 7-way single-pass helper."""
    return _infer_rows(model, tok, rows, NUM_MARKS)

# ---- Stage-A dot-threshold tuning (for INSERTION into Stage B input) ----
def tune_dot_threshold(valA, val_rows):
    P=np.concatenate([p[:,0] for _,p in valA],0)
    G=np.concatenate([gold_to_labels(r["final_text"])[1][:,M2I['.']] for r in val_rows],0)
    best_t,best_f=0.5,-1.0
    for t in np.linspace(0.20,0.80,25):
        pred=(P>=t).astype(int)
        tp=((pred==1)&(G==1)).sum(); fp=((pred==1)&(G==0)).sum(); fn=((pred==0)&(G==1)).sum()
        pr=tp/max(tp+fp,1); rc=tp/max(tp+fn,1); f=2*pr*rc/(pr+rc) if (pr+rc)>0 else 0.0
        if f>best_f: best_f,best_t=f,t
    return float(best_t), float(best_f)

# ---- the cascade: A -> insert dots -> B -> assemble 7-mark probs ----
def predict_cascade(modelA, tokA, modelB, tokB, rows, th_dot):
    Pdot  = _infer_rows(modelA, tokA, rows, len(MARKS_A))                 # (n,1)
    dotted=[]
    for (words,Pd) in Pdot:
        dotted.append([w+'.' if Pd[i,0]>=th_dot else w for i,w in enumerate(words)])
    Prest = _infer_rows(modelB, tokB, rows, len(MARKS_B), enc_words_list=dotted)  # (n,6)
    out=[]
    for (words,Pd),(w2,Pr) in zip(Pdot, Prest):
        P7=np.zeros((len(words),NUM_MARKS))
        P7[:,M2I['.']]=Pd[:,0]
        for j,m in enumerate(MARKS_B): P7[:,M2I[m]]=Pr[:,j]
        out.append((words,P7))
    return out, Pdot
print("cascade train + inference ready")


## 6. Per-class threshold tuning (free multi-label decode)

In [ ]:
# =========================================================================
#  Per-class threshold search to maximize MACRO-F1.
#  In multi-label the classes are independent, so tuning each mark's threshold
#  separately is optimal for macro-F1. Rare marks get lower thresholds (recall).
# =========================================================================
def per_class_f1(y_true, y_pred):
    tp=((y_pred==1)&(y_true==1)).sum(0)
    fp=((y_pred==1)&(y_true==0)).sum(0)
    fn=((y_pred==0)&(y_true==1)).sum(0)
    prec=tp/np.clip(tp+fp,1,None); rec=tp/np.clip(tp+fn,1,None)
    f1=np.where((prec+rec)>0, 2*prec*rec/np.clip(prec+rec,1e-9,None), 0.0)
    return f1

def tune_thresholds(val_results, val_rows):
    # stack word-level gold + probs across all val rows
    golds=[]; probs=[]
    for (words,P),r in zip(val_results,val_rows):
        _,Y=gold_to_labels(r["final_text"])
        golds.append(Y); probs.append(P)
    Yt=np.concatenate(golds,0); Pp=np.concatenate(probs,0)
    grid=np.linspace(0.10,0.90,33)
    best=np.full(NUM_MARKS,0.5)
    for m in range(NUM_MARKS):
        bf,bt=-1,0.5
        for t in grid:
            pred=(Pp[:,m]>=t).astype(int)
            tp=((pred==1)&(Yt[:,m]==1)).sum()
            fp=((pred==1)&(Yt[:,m]==0)).sum()
            fn=((pred==0)&(Yt[:,m]==1)).sum()
            prec=tp/max(tp+fp,1); rec=tp/max(tp+fn,1)
            f1=2*prec*rec/(prec+rec) if (prec+rec)>0 else 0
            if f1>bf: bf,bt=f1,t
        best[m]=bt
    return best

def probs_to_multihot(words, P, thresholds):
    pred=(P>=thresholds[None,:]).astype(np.float32)
    pred=apply_honorific_rule(words,pred)   # force -X- honorifics
    return pred
print("threshold tuning ready")

## 7. Train ensemble & validate (host metric + per-class F1)

In [ ]:
# =========================================================================
#  Orchestrate the cascade and compare to the single 7-way baseline (same run).
# =========================================================================
def _host_macro(results, rows, thresholds):
    texts=[reconstruct(words, probs_to_multihot(words,P,thresholds)) for (words,P) in results]
    vdf=pd.DataFrame([{"id":i,"text":r["text"],"final_text":r["final_text"]} for i,r in enumerate(rows)])
    sdf=pd.DataFrame({"id":range(len(rows)),"final_text":texts})
    return score(vdf.copy(), sdf.copy(), "id")

def _per_class(results, rows, thresholds):
    golds=[]; preds=[]
    for (words,P),r in zip(results,rows):
        _,Y=gold_to_labels(r["final_text"])
        golds.append(Y); preds.append(probs_to_multihot(words,P,thresholds))
    return per_class_f1(np.concatenate(golds), np.concatenate(preds))

# ---------- STAGE A: dots ----------
print(">>> Stage A (dots only)")
modelA, tokA = train_one(BACKBONE, train_rows, val_rows, DOT_IDX, inject_dots=False, tag="A/dots")
valA = _infer_rows(modelA, tokA, val_rows, len(MARKS_A))
th_dot, dotf1 = tune_dot_threshold(valA, val_rows)
print(f"    dot insert threshold = {th_dot:.3f} | Stage-A dot F1 = {dotf1:.4f}")

# ---------- STAGE B: the rest, given dots ----------
print(">>> Stage B (rest | dots)")
modelB, tokB = train_one(BACKBONE, train_rows, val_rows, REST_IDX, inject_dots=True, tag="B/rest")

# ---------- cascade decode + tune all 7 thresholds on assembled probs ----------
val_casc, _ = predict_cascade(modelA, tokA, modelB, tokB, val_rows, th_dot)
th_casc = tune_thresholds(val_casc, val_rows)
macro_casc = _host_macro(val_casc, val_rows, th_casc)
f1_casc    = _per_class(val_casc, val_rows, th_casc)

# ---------- optional single 7-way baseline ----------
if TRAIN_SINGLE_BASELINE:
    print(">>> Baseline (single 7-way)")
    modelS, tokS = train_one(BACKBONE, train_rows, val_rows, list(range(NUM_MARKS)), inject_dots=False, tag="baseline/7")
    val_base = _infer_rows(modelS, tokS, val_rows, NUM_MARKS)
    th_base  = tune_thresholds(val_base, val_rows)
    macro_base = _host_macro(val_base, val_rows, th_base)
    f1_base    = _per_class(val_base, val_rows, th_base)
else:
    macro_base=None; f1_base=None

print("\n===============  VALIDATION (host macro-F1)  ===============")
if macro_base is not None:
    print(f"  single 7-way : {macro_base:.4f}")
print(f"  cascade      : {macro_casc:.4f}" + (f"   (delta {macro_casc-macro_base:+.4f})" if macro_base is not None else ""))
print("\n  per-class F1     baseline   cascade    delta")
for i,m in enumerate(MARKS):
    b = f1_base[i] if f1_base is not None else float('nan')
    star = "  <- bottleneck" if m in ('؛','!','؟',':') else ""
    print(f"    {m:2s}   {b:8.3f} {f1_casc[i]:9.3f}  {f1_casc[i]-b:+8.3f}{star}")
print("\n  cascade thresholds:", {MARKS[i]:round(float(th_casc[i]),3) for i in range(NUM_MARKS)})


## 8. Predict test & write submission

In [ ]:
# =========================================================================
#  Predict TEST via the cascade and write submission.csv (id, final_text).
# =========================================================================
test = pd.read_csv(TEST_CSV)
id_col = "id" if "id" in test.columns else test.columns[0]
test_rows=[{"text":t} for t in test["text"].tolist()]

test_casc, _ = predict_cascade(modelA, tokA, modelB, tokB, test_rows, th_dot)
pred_texts=[reconstruct(words, probs_to_multihot(words,P,th_casc)) for (words,P) in test_casc]

submission=pd.DataFrame({id_col: test[id_col], "final_text": pred_texts})
submission.to_csv("submission.csv", index=False)
print("wrote submission.csv", submission.shape, "| cascade (dots->rest)")
print(submission.head(2).to_string())
